# 03 – Training fragment-intensity models

This notebook is the third stage of the pipeline. It consumes `spectra_with_charge.parquet`, the output of `02_precursor_charge.ipynb`, and trains models that map a peptide sequence and its precursor charge to the intensities of the singly charged b and y fragment ions observed under collision-induced dissociation (CID).

All logic resides in the `sommer` package; this notebook only selects models and training settings. The design separates three concerns:

* **Model definition** (`sommer/models/`): every model subclasses `FragmentModel` and implements `forward(tokens, precursor_charge) -> (B, 58)`. The 58 output slots enumerate b₁…b₂₉ and y₁…y₂₉ in interleaved order. Slots beyond a peptide's length are masked, and every metric is invariant to the scale of the prediction, so no output normalisation is required.
* **Training** (`sommer/training/`): one generic `Trainer` with early stopping on the validation spectral angle (SA). Loss, optimiser and learning-rate schedule are selected by name in `TrainConfig`.
* **Evaluation and tracking** (`sommer/evaluation.py`, `sommer/tracking.py`): every full run is scored with the same function, overall, per precursor charge and per (charge × length × modification) bucket, and appended to `results/run_log.jsonl` together with a checkpoint.

The test partition is never read. The loader filters it out at the Parquet level

In [1]:
from dataclasses import replace

import torch

from sommer.data import load_dataset
from sommer.models import MODELS, FragmentModel, register_model
from sommer.training import TrainConfig, Trainer, pick_device, run_experiments, search
from sommer.compare import leaderboard

print("device:", pick_device())
print("registered models:", ", ".join(sorted(MODELS)))

device: mps
registered models: PeptideEncoder, PrositLike


## 1. Data

The training partition comprises the annotated training spectra together with the spectra whose precursor charge was imputed by the CatBoost classifier of notebook 02 (`split` is null for these). Imputed spectra are never placed in the validation partition, so the validation scores are based solely on annotated charges. Setting `include_imputed=False` restricts training to annotated spectra. Encoded arrays are cached in `cache/` and invalidated automatically whenever the source file changes.

In [2]:
INCLUDE_IMPUTED = True
data = load_dataset(include_imputed=INCLUDE_IMPUTED)
print({split: len(part) for split, part in data.items()}, "| val fingerprint:", data["val"].fingerprint())

{'train': 283560, 'val': 34979} | val fingerprint: 84d1d2caea9ff588


## 2. Training configuration

`TrainConfig` collects every parameter of the optimisation procedure. A *preliminary* run trains on a reproducible subsample (by default 2,000 training and 500 validation spectra) for at most two epochs. Its purpose is to check that a model trains at all. Preliminary results are written to `results/preliminary_runs.jsonl` and never enter the leaderboard. A *full* run uses all eligible spectra, restores the parameters of the epoch with the highest validation SA, and persists a checkpoint, a training curve and the run record.

In [3]:
BASE = TrainConfig(
    loss="masked_spectral_angle",   # see sommer/training/components.py: LOSSES
    optimizer="adam",               # OPTIMIZERS
    scheduler="plateau",            # SCHEDULERS (+ scheduler_params)
    lr=1e-3, weight_decay=1e-5, batch_size=512,
    max_epochs=50, patience=5, seed=1312,
)
SMOKE = replace(BASE, preliminary=True)

## 3. Preliminary runs

Each registered architecture is trained once with deliberately small constructor arguments. The resulting SA values carry no information about final performance. They only confirm that the model honours the interface and that the loss decreases.

In [4]:
SMALL = {
    "PrositLike": {"emb": 32, "hidden": 64},
    "PeptideEncoder": {"hidden_dimensions": 64, "transformer_layers": 2, "attention_heads": 4},
}
smoke_records = run_experiments(SMALL, data, SMOKE)

PrositLike [preliminary] params=131,275 n_train=2,000 n_val=500 device=mps


epoch   1  train_loss=0.6268  val SA=0.3801  lr=1.0e-03  0s  *


epoch   2  train_loss=0.6033  val SA=0.4078  lr=1.0e-03  0s  *


best epoch 2: val SA=0.4078  pearson=0.3313  z=2: 0.4464  z=3: 0.3503  z=4: 0.2776


PeptideEncoder [preliminary] params=128,642 n_train=2,000 n_val=500 device=mps


epoch   1  train_loss=0.6124  val SA=0.3907  lr=1.0e-03  0s  *


epoch   2  train_loss=0.5985  val SA=0.4071  lr=1.0e-03  0s  *


best epoch 2: val SA=0.4071  pearson=0.2982  z=2: 0.4656  z=3: 0.3183  z=4: 0.2392


## 4. Full runs

Full runs take hours on the complete training partition. They are therefore disabled by default and are enabled by setting `RUN_FULL = True`. Each completed run is persisted immediately, so a failure in a later experiment does not discard earlier results. The settings below reproduce the strongest logged configuration of the PeptideEncoder (learning rate and weight decay from the earlier Optuna search).

In [5]:
RUN_FULL = False

FULL = replace(BASE, max_epochs=100, patience=15, batch_size=128,
               lr=6.038585076587369e-4, weight_decay=1.8421828349040193e-06)
EXPERIMENTS = {
    "PeptideEncoder": {"hidden_dimensions": 256, "transformer_layers": 6, "attention_heads": 8},
}
if RUN_FULL:
    records = run_experiments(EXPERIMENTS, data, FULL)
    for r in records:
        print(r["model"], r["run_id"], f"val SA {r['val_sa']:.4f}", r["checkpoint"])

## 5. Adding a new model

A new architecture requires only a `FragmentModel` subclass with keyword-only constructor arguments, plus the `@register_model` decorator. Placed in its own file under `sommer/models/`, it is discovered automatically. It then becomes available to the trainer, the search, checkpoint restoration and the comparison. The minimal example below, which mean-pools residue embeddings and conditions on the precursor charge, is defined inline for illustration only.

In [6]:
@register_model
class MeanPoolBaseline(FragmentModel):
    def __init__(self, width: int = 64):
        super().__init__()
        self.residue = torch.nn.Embedding(23, width, padding_idx=0)
        self.charge = torch.nn.Embedding(3, width)
        self.head = torch.nn.Sequential(torch.nn.Linear(width, width), torch.nn.ReLU(), torch.nn.Linear(width, 58))

    def forward(self, tokens, precursor_charge):
        mask = (tokens != 0).unsqueeze(-1).float()
        pooled = (self.residue(tokens) * mask).sum(1) / mask.sum(1)
        return torch.nn.functional.softplus(self.head(pooled + self.charge(self.charge_index(precursor_charge))))

_ = Trainer(SMOKE).fit("MeanPoolBaseline", data, {"width": 32})

MeanPoolBaseline [preliminary] params=3,802 n_train=2,000 n_val=500 device=mps


epoch   1  train_loss=0.6334  val SA=0.3697  lr=1.0e-03  0s  *


epoch   2  train_loss=0.6190  val SA=0.3831  lr=1.0e-03  0s  *


best epoch 2: val SA=0.3831  pearson=0.2190  z=2: 0.4213  z=3: 0.3253  z=4: 0.2714


## 6. Modifying the training algorithm

Most changes to the optimisation procedure are a single field of `TrainConfig`, e.g. `loss="masked_mse"`, `optimizer="adamw"`, `scheduler="cosine"` or `grad_clip=1.0`. A new loss, optimiser or schedule becomes available to all models once it is added to the dictionaries in `sommer/training/components.py`. Deeper changes override one step of the `Trainer` (`build_optimizer`, `build_scheduler`, `compute_loss`, `train_epoch`, `validate`). The example below re-weights the spectral-angle loss by inverse charge frequency, which raises the influence of the under-represented precursor charge 4.

In [7]:
from sommer.training.components import masked_spectral_angle_loss

class ChargeWeightedTrainer(Trainer):
    WEIGHTS = {2: 1.0, 3: 1.7, 4: 4.0}

    def compute_loss(self, model, batch):
        pred = model(batch.tokens, batch.charge)
        weights = torch.tensor([self.WEIGHTS[int(z)] for z in batch.charge], device=pred.device)
        per_spectrum = torch.stack([masked_spectral_angle_loss(p[None], t[None], m[None])
                                    for p, t, m in zip(pred, batch.target, batch.mask)])
        return (weights * per_spectrum).sum() / weights.sum()

_ = ChargeWeightedTrainer(replace(SMOKE, optimizer="adamw", scheduler="cosine")).fit("PrositLike", data, SMALL["PrositLike"])

PrositLike [preliminary] params=131,275 n_train=2,000 n_val=500 device=mps


epoch   1  train_loss=0.6426  val SA=0.3797  lr=5.0e-04  1s  *


epoch   2  train_loss=0.6246  val SA=0.3927  lr=0.0e+00  1s  *


best epoch 2: val SA=0.3927  pearson=0.3000  z=2: 0.4275  z=3: 0.3410  z=4: 0.2764


## 7. Hyperparameter search

Search strategies (`grid`, `random`) only generate candidate configurations, and every candidate is trained by the same `Trainer` in preliminary mode on an identical subsample. Repeated selection on the validation partition yields optimistically biased estimates. The selected configuration must therefore be re-trained in a full run before it appears on the leaderboard.

In [8]:
RUN_SEARCH = False

if RUN_SEARCH:
    trials = search("PrositLike", data,
                    model_space={"hidden": [64, 128, 256], "head": ["pooled", "bond"]},
                    training_space={"lr": [1e-3, 3e-4]},
                    config=replace(SMOKE, preliminary_train=20_000, preliminary_val=5_000, preliminary_epochs=5),
                    strategy="random", trials=6)
    display(trials)
    # promote: run_experiments({"PrositLike": trials.model_params[0]}, data, replace(BASE, **trials.training_overrides[0]))

## 8. Current leaderboard

Full runs on the current validation partition, ranked by mean spectral angle. A detailed comparison with earlier runs and with the Koina reference models is given in `04_compare.ipynb`.

In [9]:
_ = leaderboard(data["val"].fingerprint())

                             label            source val_set  n_params  val_sa  val_pearson    z=2    z=3    z=4             date
PeptideEncoder (legacy 2026-10-16) legacy:train_deep current   5148162  0.8552       0.9336 0.8876 0.8097 0.7439 2026-10-16 01:33
PeptideEncoder (legacy 2026-10-14)         legacy:08 current   5148162  0.8552       0.9339 0.8899 0.8065 0.7341 2026-10-14 22:56
PeptideEncoder (legacy 2026-10-14) legacy:train_deep current   5148418  0.8512       0.9310 0.8834 0.8061 0.7386 2026-10-14 01:48
